In [7]:
import pandas as pd
from pathlib import Path
import numpy as np
import json

In [17]:
DATA_DIR = Path.cwd().parent / "data"
DATA_FILE = DATA_DIR / "insurance.csv.gz"
SUBMISSION_DIR = Path.cwd().parent / "submission"
SUBMISSION_PRIVACY = SUBMISSION_DIR / "privacy_report.json"
SUBMISSION_INSURANCE = SUBMISSION_DIR / "insurance_published.csv"

In [3]:
# LEER LA BASE DE DATOS
datos = pd.read_csv(DATA_FILE, compression='gzip', encoding='utf-8')
datos

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [4]:
# Creación de nuevas columnas
datos["age_group"] = pd.cut(datos["age"],   # cut permite segmentar los datos de "age" en grupos definidos en los bins
                            bins=[18, 30, 40, 50, 65],  # Definición de los grupos de edad. Se toma 65 para que cubra a las personas con 64 años
                            labels=["18-29", "30-39", "40-49", "50-64"],   # Se agregan los labels (etiquetas) para darle un nombre a cada uno de los intervalos creados
                            right=False                 # Al definir right=False, el límite derecho de cada intervalo será abierto
)

datos["bmi_group"] = pd.cut(datos["bmi"],
                            bins=[0, 18.5, 25, 30, 100], 
                            labels=["bajo peso", "normal", "sobrepeso", "obesidad"],   
                            right=False
)

datos["children_group"] = pd.cut(datos["children"],
                                bins=[0, 1, 3, np.inf], # En este caso np.inf indica que no hay límite superior para el último conjunto: [3, ∞)
                                labels=["0", "1-2", "3+"],
                                right=False
)

datos

,age,sex,bmi,children,smoker,region,charges,age_group,bmi_group,children_group
0,19,female,27.900,0,yes,southwest,16884.92400,18-29,sobrepeso,0
1,18,male,33.770,1,no,southeast,1725.55230,18-29,obesidad,1-2
2,28,male,33.000,3,no,southeast,4449.46200,18-29,obesidad,3+
3,33,male,22.705,0,no,northwest,21984.47061,30-39,normal,0
4,32,male,28.880,0,no,northwest,3866.85520,30-39,sobrepeso,0
...,...,...,...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830,50-64,obesidad,3+
1334,18,female,31.920,0,no,northeast,2205.98080,18-29,obesidad,0
1335,18,female,36.850,0,no,southeast,1629.83350,18-29,obesidad,0
1336,21,female,25.800,0,no,southwest,2007.94500,18-29,sobrepeso,0


In [5]:
# Planteamiento de los esquemas
esquemas = {
    "with_children": ["age_group", "sex", "bmi_group", "children_group", "region"],
    "without_children": ["age_group", "sex", "bmi_group", "region"]
}

cumplen = {}  # Se crea un diccionario vacío para almacenar los resultados de cada esquema

for nombre, columnas in esquemas.items():                       # nombre corresponde a cada key del diccionario, columnas a cada lista (value)
    tamaños = datos.groupby(columnas, observed=True).size()     # Para las columnas contenidas en cada lista de esquemas se agrupan hasta analizar todas las combinaciones observadas (observed=True) y no las posibles. Al aplicar size() se identifica el número de apariciones de cada combinación observada para las columnas contenidas en cada lista
    tamaño_fila = datos.groupby(columnas, observed=True)[columnas[0]].transform("size") # Se aplica transform("size") para que el tamaño de cada combinación observada se asigne a cada fila correspondiente en la base de datos original

    condicion = tamaño_fila >= 5
    afiliados_cumplen = datos[condicion]
    cumplen[nombre] = afiliados_cumplen

data_with_children = cumplen["with_children"]
data_without_children = cumplen["without_children"]

data_with_children

,age,sex,bmi,children,smoker,region,charges,age_group,bmi_group,children_group
0,19,female,27.90,0,yes,southwest,16884.9240,18-29,sobrepeso,0
1,18,male,33.77,1,no,southeast,1725.5523,18-29,obesidad,1-2
5,31,female,25.74,0,no,southeast,3756.6216,30-39,sobrepeso,0
6,46,female,33.44,1,no,southeast,8240.5896,40-49,obesidad,1-2
8,37,male,29.83,2,no,northeast,6406.4107,30-39,sobrepeso,1-2
...,...,...,...,...,...,...,...,...,...,...
1333,50,male,30.97,3,no,northwest,10600.5483,50-64,obesidad,3+
1334,18,female,31.92,0,no,northeast,2205.9808,18-29,obesidad,0
1335,18,female,36.85,0,no,southeast,1629.8335,18-29,obesidad,0
1336,21,female,25.80,0,no,southwest,2007.9450,18-29,sobrepeso,0


In [6]:
# Planteamiento de los esquemas
esquemas = {
    "with_children": ["age_group", "sex", "bmi_group", "children_group", "region"],
    "without_children": ["age_group", "sex", "bmi_group", "region"]
}

cumplen = {}  # Se crea un diccionario vacío para almacenar los resultados de cada esquema

for nombre, columnas in esquemas.items():                       # nombre corresponde a cada key del diccionario, columnas a cada lista (value)
    tamaños = datos.groupby(columnas, observed=True).size()     # Para las columnas contenidas en cada lista de esquemas se agrupan hasta analizar todas las combinaciones observadas (observed=True) y no las posibles. Al aplicar size() se identifica el número de apariciones de cada combinación observada para las columnas contenidas en cada lista
    equivalence_classes = len(tamaños)                          # Se obtiene el número de clases de equivalencia para cada esquema
    k_before_suppression = int(tamaños.min())                   # Se obtiene el tamaño mínimo de las clases de equivalencia para cada esquema


    tamaño_fila = datos.groupby(columnas, observed=True)[columnas[0]].transform("size") # Se aplica transform("size") para que el tamaño de cada combinación observada se asigne a cada fila correspondiente en la base de datos original

    condicion = tamaño_fila >= 5
    afiliados_cumplen = datos[condicion]
    published_records = len(afiliados_cumplen)
    published_classes = len(afiliados_cumplen.groupby(columnas, observed=True).size())
    suppressed_records = len(datos) - published_records 

    data_without_smoker_diversity = afiliados_cumplen.groupby(columnas, observed=True).filter(lambda x: len(x['smoker'].unique()) < 2)
    records_without_smoker_diversity = len(data_without_smoker_diversity)
    groups_without_smoker_diversity = data_without_smoker_diversity.groupby(columnas, observed=True).size()
    classes_without_smoker_diversity = len(groups_without_smoker_diversity)

    cumplen[nombre] = {
        "data_publicada": afiliados_cumplen,
        "equivalence_classes": equivalence_classes,
        "k_before_suppression": k_before_suppression,
        "quasi_identifiers": columnas,
        "published_records": published_records,
        "suppressed_records": suppressed_records,
        "published_classes": published_classes,
        "records_without_smoker_diversity": records_without_smoker_diversity,
        "classes_without_smoker_diversity": classes_without_smoker_diversity
    }

datos_original_k = datos.groupby(["age", "sex", "bmi", "children", "region"], observed=True).size()
original_k = int(datos_original_k.min())
original_unique_records = int(sum(datos_original_k == 1))


data_with_children = cumplen["with_children"]
data_without_children = cumplen["without_children"]

In [ ]:
schemes = {}

for nombre, r in cumplen.items():
    schemes[nombre] = {k: v for k, v in r.items() if k != "data_publicada"}

selected_scheme = min(cumplen, key=lambda n: cumplen[n]["suppressed_records"])
mean_charges_original = float(datos["charges"].mean())
datos_publicados = cumplen[selected_scheme]["data_publicada"]
mean_charges_published = float(datos_publicados["charges"].mean())
smoker_rate_original = (datos["smoker"] == "yes").mean()
smoker_rate_published = (datos_publicados["smoker"] == "yes").mean()

# SEGUIR CON SELECTED SCHEME

reporte = {
    "original_k": original_k,
    "original_unique_records": original_unique_records,
    "schemes": schemes,
    "selected_scheme": selected_scheme,
    "mean_charges_original": mean_charges_original,
    "mean_charges_published": mean_charges_published,
    "smoker_rate_original": smoker_rate_original,
    "smoker_rate_published": smoker_rate_published
}

In [ ]:
with open(SUBMISSION_PRIVACY, "w") as f:
    json.dump(reporte, f, indent=4)

In [ ]:
COLUMNAS_INSURANCE_PUBLISHED = schemes[selected_scheme]["quasi_identifiers"] + ["smoker", "charges"]

datos_publicados[COLUMNAS_INSURANCE_PUBLISHED].to_csv(SUBMISSION_INSURANCE, index=False)

,age,sex,bmi,children,smoker,region,charges,age_group,bmi_group,children_group
0,19,female,27.900,0,yes,southwest,16884.92400,18-29,sobrepeso,0
1,18,male,33.770,1,no,southeast,1725.55230,18-29,obesidad,1-2
2,28,male,33.000,3,no,southeast,4449.46200,18-29,obesidad,3+
3,33,male,22.705,0,no,northwest,21984.47061,30-39,normal,0
4,32,male,28.880,0,no,northwest,3866.85520,30-39,sobrepeso,0
...,...,...,...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830,50-64,obesidad,3+
1334,18,female,31.920,0,no,northeast,2205.98080,18-29,obesidad,0
1335,18,female,36.850,0,no,southeast,1629.83350,18-29,obesidad,0
1336,21,female,25.800,0,no,southwest,2007.94500,18-29,sobrepeso,0


In [ ]:
# -----------------------------------
# REORDENAR LA ESTRUCTURA CONSTRUIDA
# -----------------------------------

# Paquetes
import pandas as pd
from pathlib import Path
import numpy as np
import json

# Carga de las rutas
DATA_DIR = Path.cwd().parent / "data"
DATA_FILE = DATA_DIR / "insurance.csv.gz"
SUBMISSION_DIR = Path.cwd().parent / "submission"
SUBMISSION_PRIVACY = SUBMISSION_DIR / "privacy_report.json"
SUBMISSION_INSURANCE = SUBMISSION_DIR / "insurance_published.csv"

# Lectura de los datos
datos = pd.read_csv(DATA_FILE, compression='gzip', encoding='utf-8')

# Creación de las nuevas columnas solicitadas
datos["age_group"] = pd.cut(datos["age"],   # cut permite segmentar los datos de "age" en grupos definidos en los bins
                            bins=[18, 30, 40, 50, 65],  # Definición de los grupos de edad. Se toma 65 para que cubra a las personas con 64 años
                            labels=["18-29", "30-39", "40-49", "50-64"],   # Se agregan los labels (etiquetas) para darle un nombre a cada uno de los intervalos creados
                            right=False                 # Al definir right=False, el límite derecho de cada intervalo será abierto
)

datos["bmi_group"] = pd.cut(datos["bmi"],
                            bins=[0, 18.5, 25, 30, 100], 
                            labels=["bajo peso", "normal", "sobrepeso", "obesidad"],   
                            right=False
)

datos["children_group"] = pd.cut(datos["children"],
                                bins=[0, 1, 3, np.inf], # En este caso np.inf indica que no hay límite superior para el último conjunto: [3, ∞)
                                labels=["0", "1-2", "3+"],
                                right=False
)

# ---------------------------------------
# ANÁLISIS DE LOS ESQUEMAS DE K-ANONIMATO
# ---------------------------------------

# Planteamiento de los esquemas indicados: with_children y without_children
esquemas = {
    "with_children": ["age_group", "sex", "bmi_group", "children_group", "region"],
    "without_children": ["age_group", "sex", "bmi_group", "region"]
}

resultados_esquemas = {}  # Se crea un diccionario vacío para almacenar los resultados de cada esquema

for nombre, columnas in esquemas.items():                                                 # En este loop nombre es la llave de cada esquema y columnas es la lista de columnas asociada a cada llave
    tamaño_x_agrupacion = datos.groupby(columnas, observed=True).size()                   # datos se agrupa según el número de columnas definidas en cada uno de los esquemas (5 y 4 columnas). size regresa para cada una de ellas el número de veces que se identificó un sujeto con una coincidencia exacta para un mismo esquema para las diferentes combinaciones posibles generadas por el groupby
    equivalence_classes= len(tamaño_x_agrupacion)                                         # Número de diferentes combinaciones obsevradas para el esquema with_children y without_children
    k_before_suppression = int(tamaño_x_agrupacion.min())                                 # Mínimo número de coincidencias para cada esquema. Es decir, el valor mínimo de k antes de aplicar la supresión                               
    tamaño_x_fila = datos.groupby(columnas, observed=True)[columnas[0]].transform("size") # Se realiza la misma agrupación que en tamaños_x_agrupacion. La diferencia es el formato en que se entrega la información. Aplicando column[0] se indicó que al análisis debe ser sobre "age_group" y con transform se hizo el conteo del número de coincidencias sobre todas las combinaciones posibles, peero asignando el resultado para cada fila (cada observación). En tamaño_x_agrupacion se tiene el número de coincidencias se tiene un resumen: el número de coincidencias por categoría: no por fila.

    condicion = tamaño_x_fila >= 5                                                           # Se aplica la condición de k-anonimato. Es decir, la condicion indica que para cada fila (observación) se deben tener al menos otras 4 coincidencias (k=5)
    observaciones_cumplen = datos[condicion]                                                 # Se seleccionan las filas que cumplen la condición de k anonimato k=5
    published_records = len(observaciones_cumplen)                                           # Número de observaciones que cumplen la condición k=5
    published_classes = len(observaciones_cumplen.groupby(columnas, observed=True).size())   # Se repite la misma agrupación realizada en tamaño por agrupacion, pero ahora se realiza sobre las filas que cumplen la condicipon de k-anonimato>=5
    suppressed_records = len(datos) - published_records                                      # Cálculo de las agrupaciones que no cumplieron la condición de k-anonimato=5. Es decir, el número de observaciones que se suprimieron

    data_without_smoker_diversity = observaciones_cumplen.groupby(columnas, observed=True).filter(lambda x: len(x['smoker'].unique()) < 2)  # x corresponde a cada una de las diferentes agrupaciones construidas. Las agrupaciones e realizaron sobre las 5 y 4 columnas indicadas en los esquemas. Smoker no es una de ellas. Para cada una de las agrupaciones de las columnas quiero validar: Tengo agrupaciones que dentro de sus filas de smoker tienen todas el mismo tipo de respuesta, ya sea "yes" o "no"? al poner la condición <2 se indica que los casos true hablan de agrupaciones con respuestas uniformes (without_diversity) y las false de with_diversity. El filter recopila los casos true y los asigna a data_without_smoker_diversity
    records_without_smoker_diversity = len(data_without_smoker_diversity)
    groups_without_smoker_diversity = data_without_smoker_diversity.groupby(columnas, observed=True).size()
    classes_without_smoker_diversity = len(groups_without_smoker_diversity)

    resultados_esquemas[nombre] = {
        "data_publicada": observaciones_cumplen,
        "equivalence_classes": equivalence_classes,
        "k_before_suppression": k_before_suppression,
        "quasi_identifiers": columnas,
        "published_records": published_records,
        "suppressed_records": suppressed_records,
        "published_classes": published_classes,
        "records_without_smoker_diversity": records_without_smoker_diversity,
        "classes_without_smoker_diversity": classes_without_smoker_diversity
    }

schemes = {}


for nombre, r in resultados_esquemas.items():
    schemes[nombre] = {k: v for k, v in r.items() if k != "data_publicada"} # Para cada uno de los dos esquemas: with_children y without_children se tiene que, nombre es el nombre del respectivo esquema y r es el diccionario con los resultados (las nueve entradas). k serían las llaves de estas nueve entradas y v sus value, finalente, se excluye data_publicada dell nuvo diccionario schemes por el formato de respuesta solicitado


# ----------------------------------------------------------------------
# ANÁLISIS DE LOS ITEMS DEL DICCIONARIO NO ANIDADOS EN SCHEMES: ESQUEMAS
# ----------------------------------------------------------------------
datos_original_k = datos.groupby(["age", "sex", "bmi", "children", "region"], observed=True).size()
original_k = int(datos_original_k.min())
original_unique_records = int(sum(datos_original_k == 1))
selected_scheme = min(resultados_esquemas, key=lambda n: resultados_esquemas[n]["suppressed_records"])  # n representa los nombres de cada uno de los esquemas: with_children y without_children. El esquema elegido es aquel con menor número de registros suprimidos
mean_charges_original = float(datos["charges"].mean())
datos_publicados = resultados_esquemas[selected_scheme]["data_publicada"]   # Del diccionario resultados_esquemas se selecciona a selected_scheme, es decir, el diccionario con menor número de registros suprimidos y luego se toma la entrada data_publicada que contiene el dataframe con los registros que cumplen con k-anonimato=5
mean_charges_published = float(datos_publicados["charges"].mean())
smoker_rate_original = (datos["smoker"] == "yes").mean()
smoker_rate_published = (datos_publicados["smoker"] == "yes").mean()

# ---------------------------
# PLANTEAR EL REPORTE FINAL
# ---------------------------
reporte = {
    "original_k": original_k,
    "original_unique_records": original_unique_records,
    "schemes": schemes,
    "selected_scheme": selected_scheme,
    "mean_charges_original": mean_charges_original,
    "mean_charges_published": mean_charges_published,
    "smoker_rate_original": smoker_rate_original,
    "smoker_rate_published": smoker_rate_published
}


# -------------------
# ARCHIVOS DE SALIDA
# -------------------
with open(SUBMISSION_PRIVACY, "w") as f:
    json.dump(reporte, f, indent=4)

COLUMNAS_INSURANCE_PUBLISHED = schemes[selected_scheme]["quasi_identifiers"] + ["smoker", "charges"]
datos_publicados[COLUMNAS_INSURANCE_PUBLISHED].to_csv(SUBMISSION_INSURANCE, index=False)